# Phase 1: Ingestion
Parse raw scorecard JSON into structured ParsedContext.

In [3]:
import sys
from pathlib import Path

ROOT = Path.cwd() if Path.cwd().name == 'certification_framework' else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [4]:
from scripts.ingestion.ingestor import ingest_from_file, save_context

input_path = ROOT / 'data' / 'input' / 'aggregated_scorecard_output.json'
NB_DATA = ROOT / 'notebooks' / 'data'
NB_DATA.mkdir(exist_ok=True)

ctx = ingest_from_file(input_path)
save_context(ctx, NB_DATA / 'parsed_context.json')

print(f'Agent:      {ctx.meta["agent_name"]}')
print(f'Agent ID:   {ctx.meta["agent_id"]}')
print(f'Date:       {ctx.meta["certification_date"]}')
print(f'Total runs: {ctx.meta["total_runs"]}')
print(f'Categories: {len(ctx.categories)}')
print(f'Warnings:   {len(ctx.warnings)}')

Agent:      Flash Agent
Agent ID:   flash-001-abc123
Date:       2026-03-08
Total runs: 15
Categories: 3
Warnings:   2


In [5]:
import json

raw = json.loads(input_path.read_text(encoding='utf-8'))
parsed = json.loads((NB_DATA / 'parsed_context.json').read_text(encoding='utf-8'))

print('=== Validation: Parsed vs Input ===')
checks = [
    ('agent_name',         parsed['meta']['agent_name'],         raw['agent_name']),
    ('agent_id',           parsed['meta']['agent_id'],           raw['agent_id']),
    ('total_runs',         parsed['meta']['total_runs'],         raw['total_runs']),
    ('total_faults',       parsed['meta']['total_faults_tested'],raw['total_faults_tested']),
    ('total_categories',   parsed['meta']['total_fault_categories'], raw['total_fault_categories']),
    ('category_count',     len(parsed['categories']),            len(raw['fault_category_scorecards'])),
]

all_ok = True
for label, got, expected in checks:
    ok = 'PASS' if got == expected else 'FAIL'
    if ok == 'FAIL':
        all_ok = False
    print(f'  {ok} {label}: {got} (expected {expected})')

# Validate per-category runs
for i, cat in enumerate(parsed['categories']):
    raw_cat = raw['fault_category_scorecards'][i]
    runs_ok = cat['total_runs'] == raw_cat['total_runs']
    ok = 'PASS' if runs_ok else 'FAIL'
    if not runs_ok:
        all_ok = False
    print(f'  {ok} {cat["label"]} runs: {cat["total_runs"]} (expected {raw_cat["total_runs"]})')

print(f'\nResult: {"ALL CHECKS PASSED" if all_ok else "SOME CHECKS FAILED"}')

=== Validation: Parsed vs Input ===
  PASS agent_name: Flash Agent (expected Flash Agent)
  PASS agent_id: flash-001-abc123 (expected flash-001-abc123)
  PASS total_runs: 15 (expected 15)
  PASS total_faults: 3 (expected 3)
  PASS total_categories: 3 (expected 3)
  PASS category_count: 3 (expected 3)
  PASS Application runs: 5 (expected 5)
  PASS Network runs: 5 (expected 5)
  PASS Resource runs: 5 (expected 5)

Result: ALL CHECKS PASSED
